In [1]:
%pip install -qU pypdf
%pip install -U langchain
%pip install -U langchain-community
%pip install -U langchain-groq
%pip install langchain-huggingface
%pip install faiss-cpu
%pip install -U langchain-google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 338.8/338.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 113.6/113.6 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 173.8/173.8 kB 11.1 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
  Attempting uninstall: langgraph-prebuilt
    Found existing installation: langgraph-prebuilt 1.0.10
    Uninstalling langgraph-prebuilt-1.0.10:
      Successfully uninstalled langgraph-prebuilt-1.0.10
  Attempting uninstall: langgraph
    Found existing installation: langgraph 1.1.9
    Uninstalling langgraph-1.1.9:
      Successfully uninstalled langgraph-1.1.9
  Attempting uninstall: langchain
    Found existing installation: langchain 1.2.15
    Uninstalling langchain-1.2.15:
      Su

In [2]:
import pandas as pd
from google.colab import userdata
import json
import os
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI

# Configuração de modelo

Aqui resgatamos chaves de API para o Groq e para o Google, e configuramos como variáveis de ambiente para permitir a comunicação com esses serviços. Em seguida instanciamos três modelos distintos usando as bibliotecas do LangChain: Llama 3.3 70B e um modelo GPT OSS de 20B (ambos através da plataforma Groq), além do Gemini 3.1 Flash Lite (através do Google).

In [3]:
os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')
os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY')

llm1 = ChatGroq(model='llama-3.3-70b-versatile')

llm2 = ChatGroq(model='openai/gpt-oss-20b')

llm3 = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")

modelos = {
    "Llama 3.3 70B": llm1,
    "GPT OSS 20B": llm2,
    "Gemini 3.1 Flash Lite": llm3
}

Por fim, agrupamos essas três conexões em um dicionário chamado *modelos*, atribuindo um nome fácil para cada um. Isso facilita a iteração no código futuro, permitindo rodar a mesma instrução em todos eles com facilidade.

# O Banco de Dados Vetorial

In [4]:
#link do arquivo: https://github.com/DMGravina/Challenge-sprint1-GENAI/blob/main/Dados_Motor.pdf
url = 'https://raw.githubusercontent.com/DMGravina/Challenge-sprint1-GENAI/main/Dados_Motor.pdf'
loader = PyPDFLoader(url)
pages = loader.load()

embed_model = HuggingFaceEmbeddings(model_name="mixedbread-ai/mxbai-embed-large-v1")

vector_db = FAISS.from_documents(pages, embed_model)
retriever = vector_db.as_retriever(search_kwargs={"k": 3})

print("Banco de dados técnico pronto para consultas!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/695 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/297 [00:00<?, ?B/s]

Banco de dados técnico pronto para consultas!


# Prompt Engineering

Nesta etapa, submetemos três modelos distintos (Llama 3.3 70B, GPT OSS 20B e Gemini 3.1 Flash Lite) a um teste de extração de informação técnica crítica: o tempo de rotor bloqueado do motor WEG W22. Utilizamos quatro técnicas de prompting para avaliar a flexibilidade e o raciocínio de cada arquitetura.

In [5]:
def prompt_tec(pergunta_tecnica):
    tecnicas = {
        "Zero-Shot": "Responda diretamente: {pergunta}",
        "Few-Shot": "Exemplo 1: Potência? R: 2.2 kW. Exemplo 2: Frequência? R: 60 Hz. Pergunta: {pergunta}",
        "Chain-of-Thought": "Pense passo a passo analisando o manual técnico e depois responda: {pergunta}",
        "Role-Playing": "Você é um Engenheiro de Manutenção Sênior da Forzy. Responda com rigor técnico: {pergunta}"
    }

    documentos = retriever.invoke(pergunta_tecnica)
    contexto = "\n".join([doc.page_content for doc in documentos])

    for nome_modelo, llm in modelos.items():
        print(f"\n{'='*50}")
        print(f"Iniciando testes com o modelo: {nome_modelo}")
        print(f"{'='*50}")

        for nome_tecnica, template in tecnicas.items():
            prompt = ChatPromptTemplate.from_template(f"Contexto técnico: {contexto}\n\n{template}")
            chain = prompt | llm
            resposta = chain.invoke({"pergunta": pergunta_tecnica})

            print(f"\n Técnica: {nome_tecnica}")
            print(resposta.content)

prompt_tec("Com base no datasheet, qual é o tempo de rotor bloqueado a frio e a quente do motor WEG W22?")


Iniciando testes com o modelo: Llama 3.3 70B

 Técnica: Zero-Shot
O tempo de rotor bloqueado a frio é de 16s e a quente é de 9s.

 Técnica: Few-Shot
De acordo com a folha de dados do motor W22 Monofásico da WEG, o tempo de rotor bloqueado é:

* A frio: 16s
* A quente: 9s

Esses valores estão listados na seção "Características Técnicas" da folha de dados.

 Técnica: Chain-of-Thought
Com base no datasheet fornecido, podemos analisar as informações técnicas do motor WEG W22 Monofásico. O tempo de rotor bloqueado é uma das especificações importantes para entender o desempenho do motor em condições de carga e temperatura.

De acordo com o manual técnico, o tempo de rotor bloqueado é fornecido para duas condições: a frio e a quente.

- **Tempo de rotor bloqueado a frio:** 16s
- **Tempo de rotor bloqueado a quente:** 9s

Isso significa que, quando o motor está frio (ou seja, não está operando há algum tempo e não está aquecido), ele pode suportar um tempo de bloqueio do rotor de 16 segundos.

- **Llama 3.3 70B** (Vencedor): Acurácia de 100%. Destacou-se nas técnicas avançadas (Chain-of-Thought e Role-Playing), seguindo instruções rigorosas e adicionando contexto real de manutenção sem inventar dados. É a escolha mais segura para gerar os dados sintéticos.

- **GPT OSS 20B** (Reprovado): Acertou o básico, mas cometeu um erro grave no Role-Playing: alucinou parâmetros físicos, inventando limites de temperatura (25°C e 125°C) que não estão no manual. Inviável para aplicações preditivas onde dados falsos podem causar falhas na planta.

- **Gemini 3.1 Flash Lite** (O segundo lugar): Acurácia de 100%. Entregou a resposta mais rica tecnicamente, cruzando os tempos de bloqueio com a "Classe de Isolamento F" do motor. Apesar da saída bruta da API exigir limpeza no código, é uma excelente opção para o assistente conversacional final.

>***Veredito***: Técnicas como Role-Playing + Chain-of-Thought geram o maior valor de negócio, mas exigem modelos robustos. O Llama 3.3 70B foi selecionado como motor principal para a próxima etapa (geração de 200 dados sintéticos) por garantir zero alucinação física, mantendo a integridade necessária para o Gêmeo Digital da Forzy.

# Geração de Dados Sintéticos

Esta célula funciona como o motor de produção do dataset sintético, utilizando o modelo Llama 3.3 70B para gerar 200 registros de manutenção técnica de forma estruturada. A lógica baseia-se em uma estratégia de geração em lote, onde um loop sequencial divide a meta total em 10 blocos de 20 registros cada.

In [6]:
prompt_sintetico = ChatPromptTemplate.from_template("""
Baseado no datasheet do motor WEG fornecido, gere 20 registros sequenciais de logs de manutenção em formato CSV.
Cada linha deve conter: Data (ano 2024, formato YYYY-MM-DD), Temperatura (°C), Vibração (mm/s), Status (Normal/Falha), Descrição Técnica.

REGRAS FÍSICAS OBRIGATÓRIAS:
- Operação Normal: Temperatura entre 50°C e 90°C. Vibração entre 0.4 e 1.8 mm/s.
- Falha por Sobreaquecimento: Temperatura entre 110°C e 145°C (A classe de isolamento F suporta até 155°C, acima disso é derretimento).
- Falha por Vibração: Vibração entre 3.5 e 7.0 mm/s (Desalinhamento grave ou rolamento quebrado).
Não inclua cabeçalhos extras. Varie os dados para ter 80% Normal e 20% Falha.
""")
dataset_chain = prompt_sintetico | llm1

todos_os_registros = []

for i in range(10):
    print(f"Gerando bloco {i+1} de 10...")
    resposta = dataset_chain.invoke({})
    todos_os_registros.append(resposta.content)

with open('dados_sinteticos_sprint1.csv', 'w', encoding='utf-8') as f:
    for bloco in todos_os_registros:
        f.write(bloco + "\n")

print("\nArquivo de 200 registros salvo com sucesso!")

Gerando bloco 1 de 10...
Gerando bloco 2 de 10...
Gerando bloco 3 de 10...
Gerando bloco 4 de 10...
Gerando bloco 5 de 10...
Gerando bloco 6 de 10...
Gerando bloco 7 de 10...
Gerando bloco 8 de 10...
Gerando bloco 9 de 10...
Gerando bloco 10 de 10...

Arquivo de 200 registros salvo com sucesso!


>Ao final da execução, todas as iterações são consolidadas em uma lista e exportadas para o arquivo dados_sinteticos_sprint1.csv, entregando uma base de dados pronta e validada para treinar os algoritmos de detecção de anomalias do Gêmeo Digital.